# Análise Exploratória dos Dados

O dataser possui 645 municípios e 80 variáveis, misturando:

- Dados geográficos (latitude, longitude)
- Infraestrutura de saúde (hospitais, clínicas, consultórios)
- Indicadores populacionais
- Procedimentos do SUS
- Receita anual
- Um índice (ids)

##### Primeiro faremos os imports e configurações necessárias para trabalhar:

In [ ]:
import sys
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import geopandas as gpd

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.config import PATHS
from src.core.transformer import purge_numbers

RAW_DIR = PATHS.data_raw
PROC_DIR = PATHS.data_processed

# Configurações visuais
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Tamanho padrão dos gráficos
plt.rcParams["figure.figsize"] = (10, 6)

##### Carregando os dados e ajuste na coluna de latitude e longitude

In [ ]:
# Lendo arquivos necessários
sp = gpd.read_file(RAW_DIR / "SP_Municipios_2025.shp")
df = pd.read_csv(PROC_DIR / "main_dataframe.csv", sep=";")

# Ajuste na formatação dos nomes vindo do shape file. 
sp["NM_MUN"] = purge_numbers(sp["NM_MUN"])

# Conversão de tipos
df["latitude"] = (df["latitude"].astype(str).str.replace(",", ".").astype(float))
df["longitude"] = (df["longitude"].astype(str).str.replace(",", ".").astype(float))

print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")
df.head()

##### Funções de Plotagem

In [ ]:
# Histograma
def plot_histogram(dataframe, coluna, bins=30):

    plt.figure(figsize=(8,4))
    sns.histplot(dataframe[coluna], bins=bins, kde=True)
    plt.title(f"Distribuição de {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.show()

# Boxplot
def plot_boxplot(dataframe, coluna):

    plt.figure(figsize=(9,5))
    sns.boxplot(x=dataframe[coluna])
    plt.title(f"Boxplot - {coluna}")
    plt.show()

# Scatter
def plot_scatter(dataframe, coluna1, coluna2):
    plt.figure(figsize=(7,5))
    plt.scatter(data=df, x=coluna1, y=coluna2, s=10)
    plt.title(f"{coluna1} x {coluna2}")
    plt.show()
    
# Correlação
def plot_correlation(dataframe, tamanho=(14,10)):

    plt.figure(figsize=tamanho)
    corr = dataframe.corr(numeric_only=True)
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Matriz de Correlação")
    plt.show()

# Top Municípios
def plot_top_municipios(dataframe, coluna, top_n=15):

    top = (dataframe.sort_values(coluna, ascending=False).head(top_n))

    plt.figure(figsize=(12,6))
    sns.barplot(data=top, x=coluna, y="municipio")
    plt.title(f"Top {top_n} municípios - {coluna}")
    plt.show()

##### Visão Geral do Dataset

In [ ]:
df.info()

# Distribuição das Variáveis

### Municípios

In [ ]:
plot_scatter(df, 'longitude', 'latitude')

### População

In [ ]:
plot_boxplot(df, "populacao")

### Receita

In [ ]:
plot_histogram(df, "receita_anual")
plot_boxplot(df, "receita_anual")

### IDS

In [ ]:
plot_histogram(df, "ids")
plot_boxplot(df, "ids")

### Correlação de Variáveis

In [ ]:
plot_correlation(df)

##### Correlação com o IDS

In [ ]:
corr_ids = (df.corr(numeric_only=True)["ids"].sort_values(ascending=False))
corr_ids.head(20)

In [ ]:
corr_ids.tail(20)

##### População Vs Receita

In [ ]:
plot_scatter(df, "populacao", "receita_anual")

### Municipios com maior receita

In [ ]:
plot_top_municipios(df, "receita_anual")

### Municípios com maior IDS

In [ ]:
plot_top_municipios(df, "ids")

### Top municípios com mais atendimentos

In [ ]:
if 'consultas / atendimentos / acompanhamentos' in df.columns:
    top_municipios = df.sort_values(by='consultas / atendimentos / acompanhamentos', ascending=False)
    print(top_municipios[['municipio', 'consultas / atendimentos / acompanhamentos']].head(10))


### Top municípios com mais hospitais

In [ ]:
df['total_hospitais'] = (df['hospital_estadual'] + df['hospital_federal'] + df['hospital_municipal'] + df['hospital_particular'])
top_hospitais = df.sort_values(by='total_hospitais', ascending=False)

print(top_hospitais[['municipio', 'total_hospitais']].head(10))

### Top municípios com mais clínicas

In [ ]:
df['total_clinicas'] = (df['clinica_estadual'] + df['clinica_federal'] + df['clinica_municipal'] + df['clinica_particular'])
top_clinicas = df.sort_values(by='total_clinicas', ascending=False)

print(top_clinicas[['municipio', 'total_clinicas']].head(10))

### Top municípios com maior vulnerabilidade social

In [ ]:
vulnerabilidade_cols = [
    'baixissima vulnerabilidade_n_pessoas',
    'muito baixa vulnerabilidade_n_pessoas',
    'baixa vulnerabilidade_n_pessoas',
    'media vulnerabilidade_n_pessoas',
    'alta vulnerabilidade_n_pessoas',
    'muito alta vulnerabilidade_n_pessoas'
]

df['vulnerabilidade_total'] = df[vulnerabilidade_cols].sum(axis=1)
top_vulnerabilidade = df.sort_values(by='vulnerabilidade_total', ascending=False)

print(top_vulnerabilidade[['municipio', 'vulnerabilidade_total']].head(10))

### Relação entre vulnerabilidade e atendimentos

In [ ]:
plot_scatter(df, 'vulnerabilidade_total', 'consultas / atendimentos / acompanhamentos')

### Municípios que não tem hospitais

In [ ]:
sem_hospital = df[df['total_hospitais'] == 0]
print(sem_hospital['municipio'].head(20))

print(f"\nTotal: {len(sem_hospital)}")

### Renda Per Capita

In [ ]:
df['valor_per_capita'] = (df['receita_anual'] / df['populacao'])

print(df[['municipio', 'valor_per_capita']].sort_values(by='valor_per_capita',ascending=True).head(10))

### Vulnerabilidade e Gasto Público

In [ ]:
top_vulneravel = df.loc[df['vulnerabilidade_total'].idxmax()]
top_valorPerCapita = df.loc[ df['valor_per_capita'].idxmax()]

plt.figure(figsize=(8,4))
plt.scatter(df['vulnerabilidade_total'], df['valor_per_capita'])
plt.annotate(top_vulneravel['municipio'], (top_vulneravel['vulnerabilidade_total'], 
                                           top_vulneravel['valor_per_capita']), fontsize=10)

plt.annotate(top_valorPerCapita['municipio'], (top_valorPerCapita['vulnerabilidade_total'], 
                                               top_valorPerCapita['valor_per_capita']), fontsize=10)

plt.xlabel('Vulnerabilidade')
plt.ylabel('Valor Per Capita')
plt.title('Vulnerabilidade x Investimento Per Capita')

plt.show()

### Mapas cloropéticos

Primeiro vamos precisar combinar o Shapefile com o nosso dataset fazendo o merge.

In [ ]:
mapa = sp.merge(df, left_on="NM_MUN", right_on="municipio", how="left")

### MAPA IDS

In [ ]:
fig = px.choropleth(mapa, geojson=mapa.geometry, locations=mapa.index, color="ids", color_continuous_scale="Viridis", hover_name="municipio",
                     hover_data={"ids":":.3f"})

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(title="IDS dos Municípios do Estado de São Paulo", width=1200, height=900)

fig.show()

### Receita Per Capita

In [ ]:
mapa.plot(column="valor_per_capita", cmap="inferno", scheme="quantiles", linewidth=0.2, edgecolor="gray", k=5, legend=True)

### Hospitais por 10.000 habitantes

In [ ]:
fig, ax = plt.subplots(figsize=(14, 14))
mapa.plot(column=(df["total_hospitais"] / df["populacao"]) * 10000, cmap="OrRd", legend=True, linewidth=0.2, edgecolor="gray", ax=ax)

mais_hospitais = mapa.loc[((df["total_hospitais"] / df["populacao"]) * 10000).nlargest(4).index]

for n, row in mais_hospitais.iterrows():
    centroid = row.geometry.centroid

    ax.annotate(row["NM_MUN"], (centroid.x, centroid.y), fontsize=10, fontweight="bold")

ax.set_title("Hospitais por 10 mil habitantes", fontsize=18)
ax.axis("off")

plt.show()

IDS vs Infraestrutura

Quão eficiente é um município em transformar sua infraestrutura de saúde em resultados (IDS)?

Maior eficiência na gestão;
Melhor distribuição dos recursos;
Ou simplesmente que a métrica de infraestrutura não captura toda a realidade.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 14))
mapa.plot(column=(df["ids"] / (df["total_hospitais"] + df["total_clinicas"] + 1)), cmap="cividis", legend=True, linewidth=0.2, edgecolor="gray", ax=ax)
ax.set_title("IDS por Infraestrutura", fontsize=18)
ax.axis("off")

Vamos normalizar e reduzir a dimensionalidade para visualizar os dados e verificar se conseguimos novos insights.

In [ ]:
n_df = df.drop(['municipio','latitude', 'longitude', 'ids'], axis=1).select_dtypes(include="number")
normalized_df = StandardScaler().fit_transform(n_df)

pca = PCA(n_components=20)
X_pca = pca.fit_transform(normalized_df)

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=42)

X_tsne = tsne.fit_transform(X_pca)

df["TSNE1"] = X_tsne[:,0]
df["TSNE2"] = X_tsne[:,1]

In [ ]:
plt.figure(figsize=(12,8))

fig = px.scatter(
    df,
    x="TSNE1",
    y="TSNE2",
    color="ids",
    color_continuous_scale="Viridis",
    hover_name="municipio",
    hover_data={
        "ids":":.3f",
        "populacao":True,
        "receita_anual":":,.0f",
        "TSNE1":False,
        "TSNE2":False
    },
    width=1200,
    height=800
)

fig.update_layout(title="t-SNE colorido pelo IDS")

fig.show()

Vamos clusterizar e visualizar novamente.

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)

df["cluster"] = kmeans.fit_predict(X_pca)

In [ ]:
# Convertendo o tipo para string, para não influenciar na cor da plotagem
df["cluster"] = df["cluster"].astype(str)

plt.figure(figsize=(12,8))

fig = px.scatter(df, x="TSNE1", y="TSNE2", color="cluster",
    color_discrete_map={
        "0": "#1f77b4",
        "1": "#ff7f0e",
        "2": "#2ca02c",
        "3": "#d62728",
        "4": "#9467bd"
    },
    hover_name="municipio", width=1200, height=800)

fig.update_layout(
    title="Clusters visualizados com t-SNE"
)